In [2]:
import os
import boto3
import csv
import io
from dotenv import load_dotenv
from agents import Agent, Runner
from pydantic import BaseModel
from typing import Literal

load_dotenv()

BUCKET = "cultivate-mapping-data"
CITYLIST_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/CityList.csv"
SCRAPED_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/scraped_text.csv"
OUTPUT_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/llm_classification_scraped_text.csv"

MODEL = os.environ.get("OPENAI_MODEL", "gpt-5-nano")

s3 = boto3.client("s3", region_name="eu-north-1")

In [3]:
# load city → country mapping
body = s3.get_object(Bucket=BUCKET, Key=CITYLIST_KEY)["Body"].read().decode("utf-8-sig")
reader = csv.DictReader(io.StringIO(body))
city_country = {row["City"]: row["Country"] for row in reader if row.get("City")}
print(f"Loaded {len(city_country)} city-country mappings")

Loaded 200 city-country mappings


In [4]:
# load scraped text (only rows with non-empty text)
body = s3.get_object(Bucket=BUCKET, Key=SCRAPED_KEY)["Body"].read().decode("utf-8")
reader = csv.DictReader(io.StringIO(body))
scraped = [(row["city"], row["name"], row["url"], row["scraped_text"]) 
           for row in reader if row["scraped_text"].strip()]
print(f"URLs with scraped text: {len(scraped)}")

URLs with scraped text: 4737


In [8]:
# resume from checkpoint if exists
done_urls = set()
try:
    body = s3.get_object(Bucket=BUCKET, Key=OUTPUT_KEY)["Body"].read().decode("utf-8")
    reader = csv.DictReader(io.StringIO(body))
    done_urls = {row["url"] for row in reader}
    print(f"Found checkpoint: {len(done_urls)} URLs already classified")
except s3.exceptions.NoSuchKey:
    print("No checkpoint found, starting fresh")

remaining = [item for item in scraped if item[2] not in done_urls]
print(f"Remaining: {len(remaining)} URLs")

No checkpoint found, starting fresh
Remaining: 4737 URLs


In [9]:
class Classification(BaseModel):
    valid: bool
    confidence: Literal["low", "medium", "high"]
    reason: str


def build_instructions(city: str, country: str) -> str:
    return f"""
Task: Classify whether the given webpage text represents a valid Food Sharing Initiative (FSI) located in {city}, {country}.

You will receive the URL and pre-scraped text from the webpage. Base your classification strictly on this text.

Method (Strict Evidence-Based Evaluation):
1. Extract only explicit textual evidence from the provided text.
   Treat all claims as unverified unless directly supported by the text.

2. Do not infer or assume intentions based on tone, imagery, branding, or general community-oriented language.

3. Before producing output, internally evaluate (do not reveal this reasoning):
   - each VALID criterion as Confirmed, Contradicted, or Not found
   - each INVALID trigger
   - final decision based strictly on explicit evidence

4. If the text is insufficient, blank, or irrelevant, classify as INVALID with low confidence.

VALID FSI — All Six Conditions Must Be Explicitly Confirmed:

1. Direct representation:
   The website is owned by or officially represents the initiative, not a media site, directory, listing, or article.

2. Explicit food-sharing purpose:
   The site clearly states that the initiative performs food redistribution or communal food sharing, such as:
   - food rescue
   - free or pay-what-you-can meals
   - community kitchens
   - shared gardens where harvest is distributed or collectively available
   - seed/produce swaps
   - non-commercial food cooperatives
   General mentions of sustainability, community, or ecology are insufficient without explicit food-sharing activities.

3. Active food-sharing operations:
   Evidence shows ongoing or recurring food distribution or communal food-sharing activities (not a one-time event).

4. Non-commercial:
   The initiative's primary purpose is community food access, not sales or profit-making.

5. Location match:
   The initiative explicitly states an address or operational activity in {city}, {country}.
   Nearby cities, regions, or generic national presence do not qualify.

6. Evidence of continuity:
   Clear indication of recurring or ongoing programs (events, schedules, regular services).

If any required condition is "Not found", classify as INVALID.

INVALID FSI — Any One of These Is Sufficient:

- News, media, editorial, or blog content describing an initiative.
- Government or municipal pages listing external programs without operating them.
- Crowdfunding or campaign-only pages (GoFundMe, Kickstarter, etc.).
- Social media profiles without a verifiable organizational website.
- Restaurants, cafes, or food-delivery or food-retail businesses.
- Schools or cultural institutions hosting only one-off food events.
- Gardening, farming, ecology, or sustainability groups without explicit food sharing or redistribution.
- Multi-city or international networks without confirmed operations in {city}, {country}.
- Any page with insufficient evidence, ambiguous purpose, or inaccessible/empty content.

Edge Cases:

- Community centers, libraries, churches: Valid only if food sharing is a core, ongoing activity explicitly stated.
- Gardening groups: Valid only if the harvest is shared or redistributed, not merely grown.
- Coalitions or networks: Valid only if they coordinate actual food-sharing activity, not just advocacy or education.

Confidence Scoring:

- High: All required evidence is explicit, consistent, and unambiguous.
- Medium: Most evidence is explicit, but one secondary attribute (e.g., frequency or non-commercial nature) is implied.
- Low: Missing or ambiguous evidence; unclear location; partial page retrieval; or overall uncertainty.

Reason:
- Provide a concise 1-2 sentence explanation citing the specific criteria that were confirmed or missing.
"""


def make_agent(city: str, country: str) -> Agent:
    return Agent(
        name="FSI Classifier (text-based)",
        model=MODEL,
        instructions=build_instructions(city, country),
        output_type=Classification,
    )


async def classify(city: str, country: str, url: str, text: str) -> Classification:
    agent = make_agent(city, country)
    user_input = f"URL: {url}\n\nScraped text:\n{text}"
    result = await Runner.run(agent, user_input)
    return result.final_output

In [10]:
# test with first 3 URLs
test_batch = remaining[:3]
for city, name, url, text in test_batch:
    country = city_country.get(city, "")
    try:
        result = await classify(city, country, url, text)
        print(f"{city} | {name} | {url[:60]}")
        print(f"  → valid={result.valid}, confidence={result.confidence}")
        print(f"  reason: {result.reason}")
    except Exception as e:
        print(f"{city} | {url[:60]} | ERROR: {e}")

Nairobi | Frī mā mīlā | https://www.facebook.com/freemymealKENYA
  → valid=False, confidence=low
  reason: Insufficient evidence of an FSI: the scraped text only shows a Facebook page name 'Free My Meal KE | Nairobi' with no explicit statement of food-sharing activities, ongoing operations, non-commercial status, or a Nairobi address.
Nairobi | Soko la Wakulima wa Nairobi | https://www.worldfarmersmarketscoalition.org/nairobi-farmers
  → valid=False, confidence=low
  reason: The text does not contain explicit evidence of direct representation by the initiative (it appears as a World Farmers Markets Coalition page describing a Nairobi Farmers Market, not an official Nairobi FSI site). It also lacks explicit food-sharing/redistribution activities (it focuses on a market for selling local produce). It mentions Nairobi and a future opening but does not demonstrate ongoing, non-commercial, recurring food-sharing programs.
Nairobi | Amana | https://fspnafrica.org/nairobi-amana-csa-day
  → va

In [11]:
# helper: append single row to CSV in S3
def append_result(city, name, url, valid, confidence, reason="", error=""):
    try:
        body = s3.get_object(Bucket=BUCKET, Key=OUTPUT_KEY)["Body"].read().decode("utf-8")
        existing = body
    except s3.exceptions.NoSuchKey:
        existing = "city,name,url,valid,confidence,reason,error\n"

    output = io.StringIO(existing)
    output.seek(0, io.SEEK_END)
    writer = csv.writer(output)
    writer.writerow([city, name, url, valid, confidence, reason, error])
    s3.put_object(Bucket=BUCKET, Key=OUTPUT_KEY, Body=output.getvalue().encode("utf-8"))

In [ ]:
# run all remaining URLs with incremental save
for i, (city, name, url, text) in enumerate(remaining):
    country = city_country.get(city, "")
    try:
        result = await classify(city, country, url, text)
        append_result(city, name, url, result.valid, result.confidence, result.reason)
        print(f"[{i+1}/{len(remaining)}] {city} | {url[:60]} → valid={result.valid}, conf={result.confidence}")
    except Exception as e:
        append_result(city, name, url, "", "", "", str(e)[:200])
        print(f"[{i+1}/{len(remaining)}] {city} | {url[:60]} → ERROR: {e}")

print("Done!")

[1/4737] Nairobi | https://www.facebook.com/freemymealKENYA → valid=False, conf=low
[2/4737] Nairobi | https://www.worldfarmersmarketscoalition.org/nairobi-farmers → valid=False, conf=low
[3/4737] Nairobi | https://fspnafrica.org/nairobi-amana-csa-day → valid=False, conf=low
[4/4737] Nairobi | https://coamana.com/celebrating-excellence-reliving-the-csa- → valid=False, conf=low
[5/4737] Nairobi | https://cshepkenya.org → valid=False, conf=low
[6/4737] Nairobi | https://muslimfoodbank.com/kenya → valid=False, conf=low
[7/4737] Nairobi | https://www.joy-divine.com → valid=False, conf=low
